# Módulo 06 · Aula 02 — Pydantic e Rotas

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"A API do app já está integrada. Só que ontem alguém cadastrou um produto com preço `-100` e o relatório de faturamento ficou negativo. E hoje descobri que a resposta de `/produtos` estava devolvendo o **custo** e o **fornecedor** — o marketplace concorrente tem acesso a essa API."*
> — Sua chefe, às 8h da manhã

Dois problemas, uma raiz: **a API não tem contrato**. Aceita qualquer coisa que entra e devolve tudo que tem.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | Request body | Como o dado entra |
| 2 | Validação com Pydantic | 🔒 A fronteira do sistema |
| 3 | Modelos aninhados | Corpos com estrutura |
| 4 | **`response_model`** | 🔒 Controlar o que sai |
| 5 | `HTTPException` e handlers | Erros úteis, não genéricos |
| 6 | Documentar erros | O OpenAPI completo |

> 💭 **Você já conhece o Pydantic** da aula 04_04. Aqui ele muda de papel: lá era validação de dados internos, aqui é **defesa de perímetro**. É a mesma ferramenta, com muito mais em jogo.

## ⚙️ Preparação

Mesmo `TestClient` da aula anterior. **Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 06
# ═══════════════════════════════════════════════════════════════
import json
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


for _pacote, _mod in [("fastapi", "fastapi"), ("httpx", "httpx"),
                      ("uvicorn[standard]", "uvicorn"), ("pydantic", "pydantic")]:
    _garantir(_pacote, _mod)

import fastapi
from fastapi.testclient import TestClient

print(f"✅ FastAPI {fastapi.__version__}")
print(f"✅ Pydantic {__import__('pydantic').VERSION}")


# ═══════════════════════════════════════════════════════════════
#  Função auxiliar: exibe requisição e resposta lado a lado
# ═══════════════════════════════════════════════════════════════

def req(cliente, metodo: str, caminho: str, mostrar_corpo=True, **kwargs):
    """Faz uma requisição e imprime o resultado de forma legível."""
    resposta = getattr(cliente, metodo.lower())(caminho, **kwargs)

    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    print(f"{cor} {metodo.upper():<7} {caminho:<42} → {resposta.status_code}")

    if kwargs.get("json") is not None:
        corpo = json.dumps(kwargs["json"], ensure_ascii=False)
        print(f"   envio  : {corpo[:150]}{'...' if len(corpo) > 150 else ''}")

    if mostrar_corpo:
        try:
            dados = resposta.json()
            texto = json.dumps(dados, ensure_ascii=False, indent=2)
            linhas = texto.splitlines()
            for linha in linhas[:14]:
                print(f"   {linha}")
            if len(linhas) > 14:
                print(f"   ... (+{len(linhas) - 14} linhas)")
        except Exception:
            corpo = resposta.text[:200]
            if corpo.strip():
                print(f"   {corpo}")
    print()
    return resposta


print("✅ Função auxiliar `req(cliente, metodo, caminho, ...)` pronta")

## 1. O corpo da requisição

Até aqui só recebemos dados pela URL. Para **criar** e **alterar**, o dado vem no corpo, em JSON.

**A regra do FastAPI:** um parâmetro anotado com um modelo Pydantic vem do **corpo**. Os demais vêm da URL.

| Anotação | De onde vem |
|----------|-------------|
| Está no caminho da rota | Path |
| Tipo simples (`int`, `str`, `bool`) | Query |
| **Modelo Pydantic** | **Body** |
| `= Body(...)` | Body, mesmo sendo simples |

In [ ]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

app = FastAPI(title="Atlas API")


class ProdutoEntrada(BaseModel):
    sku: str = Field(min_length=5, max_length=20)
    nome: str = Field(min_length=3, max_length=120)
    categoria: str
    preco: float = Field(gt=0)
    custo: float = Field(ge=0)
    estoque: int = Field(default=0, ge=0)


CATALOGO: dict[str, dict] = {}


@app.post("/produtos", status_code=status.HTTP_201_CREATED)
def criar_produto(produto: ProdutoEntrada):
    """`produto` é um modelo Pydantic → vem do CORPO."""
    if produto.sku in CATALOGO:
        raise HTTPException(status.HTTP_409_CONFLICT, f"SKU {produto.sku} já existe")
    CATALOGO[produto.sku] = produto.model_dump()
    return CATALOGO[produto.sku]


cliente = TestClient(app)

req(cliente, "POST", "/produtos", json={
    "sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15",
    "categoria": "Notebooks", "preco": 2599.90, "custo": 2120.00, "estoque": 14,
})

In [ ]:
# 🎯 A validação acontece ANTES de a sua função rodar
req(cliente, "POST", "/produtos", json={
    "sku": "AB", "nome": "X", "categoria": "Teste",
    "preco": -100, "custo": 50, "estoque": -5,
})

> 💡 **Repare: quatro erros, todos de uma vez, cada um localizado no campo.**
>
> Uma validação manual com `if/raise` pararia no primeiro. Para quem consome a API, receber os quatro problemas juntos é a diferença entre uma tentativa e quatro.
>
> E note o `loc: ["body", "sku"]` — o cliente sabe exatamente onde está o problema.

In [ ]:
# Campo faltando e tipo errado
print("── campo obrigatório ausente ──")
r = req(cliente, "POST", "/produtos",
        json={"sku": "MO-LG-24", "nome": "Monitor LG", "custo": 900}, mostrar_corpo=False)
for e in r.json()["detail"]:
    print(f"   {'.'.join(map(str, e['loc'])):<20} {e['msg']}")

print("\n── tipo incompatível ──")
r = req(cliente, "POST", "/produtos",
        json={"sku": "MO-LG-24", "nome": "Monitor LG", "categoria": "Monitores",
              "preco": "caro", "custo": 900}, mostrar_corpo=False)
for e in r.json()["detail"]:
    print(f"   {'.'.join(map(str, e['loc'])):<20} {e['msg']}")

In [ ]:
# Conversão automática: o Pydantic aceita o que consegue converter
r = req(cliente, "POST", "/produtos", json={
    "sku": "MO-LG-24UW", "nome": "Monitor LG 24", "categoria": "Monitores",
    "preco": "1199.00",     # string → float
    "custo": 920,           # int → float
    "estoque": "31",        # string → int
})
print("Tipos após a conversão:")
for campo, valor in r.json().items():
    print(f"   {campo:<12} {type(valor).__name__:<8} {valor!r}")

## 2. Validação de verdade

Você viu o Pydantic na aula 04_04. Aqui ele ganha um papel novo: **defender a fronteira do sistema**.

> 🎯 **Todo dado que entra na API é hostil até prova em contrário.**
>
> Não porque o usuário seja mal-intencionado (embora possa ser), mas porque cliente errado, integração desatualizada e bug de terceiro existem. A validação na fronteira é o que impede que dado ruim entre no seu banco.

In [ ]:
from datetime import date
from decimal import Decimal
from enum import Enum
from typing import Annotated

from pydantic import BaseModel, Field, field_validator, model_validator


class Categoria(str, Enum):
    NOTEBOOKS = "Notebooks"
    MONITORES = "Monitores"
    PERIFERICOS = "Periféricos"
    ARMAZENAMENTO = "Armazenamento"


class ProdutoCompleto(BaseModel):
    sku: str = Field(
        min_length=5, max_length=20,
        pattern=r"^[A-Z]{2}-[A-Z0-9]+(-[A-Z0-9]+)?$",
        description="Formato: XX-YYYY ou XX-YYYY-ZZ",
        examples=["NB-DELL-15"],
    )
    nome: str = Field(min_length=3, max_length=120)
    categoria: Categoria                       # 🔒 domínio fechado
    preco: float = Field(gt=0, le=1_000_000)
    custo: float = Field(ge=0)
    estoque: int = Field(default=0, ge=0, le=100_000)
    tags: list[str] = Field(default_factory=list, max_length=10)
    ativo: bool = True

    # 🎯 mode="before": roda ANTES das restrições do Field().
    #    Sem isso, "  nb-dell-15  " seria rejeitado pelo `pattern` e o
    #    validador nem chegaria a rodar. Veja a demonstração logo abaixo.
    @field_validator("sku", mode="before")
    @classmethod
    def sku_maiusculo(cls, v: str) -> str:
        """Validadores também NORMALIZAM."""
        return v.strip().upper() if isinstance(v, str) else v

    @field_validator("nome")
    @classmethod
    def nome_limpo(cls, v: str) -> str:
        return " ".join(v.split())

    @field_validator("tags")
    @classmethod
    def tags_normalizadas(cls, v: list[str]) -> list[str]:
        return sorted({t.strip().lower() for t in v if t.strip()})

    @model_validator(mode="after")
    def preco_acima_do_custo(self):
        """Validador de MODELO — enxerga todos os campos juntos."""
        if self.preco < self.custo:
            raise ValueError(f"preço {self.preco} abaixo do custo {self.custo}")
        return self


app = FastAPI()


@app.post("/produtos", status_code=201)
def criar(produto: ProdutoCompleto):
    return produto


cliente = TestClient(app)

# Entrada suja: será normalizada
req(cliente, "POST", "/produtos", json={
    "sku": "  nb-dell-15  ",
    "nome": "Notebook   Dell    Inspiron 15",
    "categoria": "Notebooks",
    "preco": 2599.90, "custo": 2120.00,
    "tags": ["  GAMER ", "leve", "gamer", ""],
})

In [ ]:
# ⚠️ A ARMADILHA DA ORDEM: `mode="before"` vs `mode="after"`
from pydantic import ValidationError


class ComAfter(BaseModel):
    """Validador padrão (mode="after") — roda DEPOIS do pattern."""
    sku: str = Field(pattern=r"^[A-Z]{2}-[A-Z0-9-]+$")

    @field_validator("sku")
    @classmethod
    def normalizar(cls, v): return v.strip().upper()


class ComBefore(BaseModel):
    """mode="before" — roda ANTES do pattern."""
    sku: str = Field(pattern=r"^[A-Z]{2}-[A-Z0-9-]+$")

    @field_validator("sku", mode="before")
    @classmethod
    def normalizar(cls, v): return v.strip().upper()


suja = "  nb-dell-15  "
for modelo in (ComAfter, ComBefore):
    try:
        print(f"   {modelo.__name__:<10} → {modelo(sku=suja).sku!r}")
    except ValidationError as erro:
        print(f"   {modelo.__name__:<10} → 🔴 {erro.errors()[0]['msg']}")

> ⚠️ **Essa é a armadilha número um de quem começa com Pydantic.**
>
> As restrições declaradas em `Field()` (`pattern`, `min_length`, `gt`) fazem parte da **validação central**, que roda entre o `before` e o `after`:
>
> ```
> dado bruto → [mode="before"] → tipo + restrições do Field → [mode="after"] → modelo pronto
> ```
>
> **Se o validador conserta o dado, ele tem que rodar antes** — senão a restrição rejeita o dado sujo e o conserto nunca acontece.
>
> 🧭 **Regra prática:** normalizar → `mode="before"`. Verificar regra de negócio → `mode="after"` (padrão), onde o valor já chega com o tipo certo.

In [ ]:
print("── SKU fora do padrão ──")
r = req(cliente, "POST", "/produtos", json={
    "sku": "notebook1", "nome": "Teste", "categoria": "Notebooks",
    "preco": 100, "custo": 50}, mostrar_corpo=False)
print("   ", r.json()["detail"][0]["msg"])

print("\n── categoria fora do enum ──")
r = req(cliente, "POST", "/produtos", json={
    "sku": "XX-TESTE", "nome": "Teste", "categoria": "Brinquedos",
    "preco": 100, "custo": 50}, mostrar_corpo=False)
print("   ", r.json()["detail"][0]["msg"])

print("\n── preço abaixo do custo (validador de modelo) ──")
r = req(cliente, "POST", "/produtos", json={
    "sku": "XX-TESTE", "nome": "Teste", "categoria": "Notebooks",
    "preco": 50, "custo": 100}, mostrar_corpo=False)
print("   ", r.json()["detail"][0]["msg"])

> ⚠️ **`field_validator` vs `model_validator`:**
>
> | | Enxerga | Use para |
> |---|---------|----------|
> | `field_validator` | **Um** campo | Formato, normalização, faixa |
> | `model_validator(mode="after")` | **Todos** os campos | Regra que cruza campos |
>
> `preco >= custo` precisa dos dois valores — por isso é de modelo.
>
> ⚠️ **Um validador de campo só roda se aquele campo passou na validação básica.** Se `preco` falhou no `gt=0`, o `model_validator` nem executa. Você viu isso no M04.

## 3. Modelos aninhados

O corpo pode ter estrutura. O Pydantic valida em profundidade.

In [ ]:
class ItemPedido(BaseModel):
    sku: str = Field(min_length=5)
    quantidade: int = Field(gt=0, le=1000)
    preco_unitario: float = Field(gt=0)

    @property
    def total(self) -> float:
        return round(self.quantidade * self.preco_unitario, 2)


class Endereco(BaseModel):
    logradouro: str
    cidade: str
    uf: str = Field(min_length=2, max_length=2, pattern=r"^[A-Z]{2}$")
    cep: str = Field(pattern=r"^\d{5}-?\d{3}$")

    @field_validator("uf", mode="before")
    @classmethod
    def uf_maiuscula(cls, v):
        return v.upper() if isinstance(v, str) else v


class PedidoEntrada(BaseModel):
    cliente_email: str = Field(pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    canal: str = Field(pattern="^(site|app|marketplace)$")
    entrega: Endereco
    itens: list[ItemPedido] = Field(min_length=1, max_length=50)
    observacao: str | None = Field(default=None, max_length=500)

    @model_validator(mode="after")
    def sem_sku_repetido(self):
        skus = [i.sku for i in self.itens]
        if len(skus) != len(set(skus)):
            raise ValueError("o mesmo SKU aparece mais de uma vez")
        return self


app = FastAPI()


@app.post("/pedidos", status_code=201)
def criar_pedido(pedido: PedidoEntrada):
    return {
        "id": 9001,
        "cliente": pedido.cliente_email,
        "praca": f"{pedido.entrega.cidade}/{pedido.entrega.uf}",
        "itens": len(pedido.itens),
        "total": round(sum(i.total for i in pedido.itens), 2),
    }


cliente = TestClient(app)

req(cliente, "POST", "/pedidos", json={
    "cliente_email": "maria@aurora.com.br",
    "canal": "site",
    "entrega": {"logradouro": "Rua das Flores, 100", "cidade": "Campinas",
                "uf": "sp", "cep": "13010-000"},
    "itens": [
        {"sku": "NB-DELL-15", "quantidade": 2, "preco_unitario": 2599.90},
        {"sku": "MO-LG-24UW", "quantidade": 1, "preco_unitario": 1199.00},
    ],
})

In [ ]:
# Erro em campo ANINHADO — repare no `loc`
r = req(cliente, "POST", "/pedidos", json={
    "cliente_email": "email-invalido",
    "canal": "telefone",
    "entrega": {"logradouro": "Rua X", "cidade": "Campinas", "uf": "SPP", "cep": "13010"},
    "itens": [{"sku": "NB-DELL-15", "quantidade": 0, "preco_unitario": 2599.90},
              {"sku": "NB-DELL-15", "quantidade": 1, "preco_unitario": 100}],
}, mostrar_corpo=False)

print(f"{len(r.json()['detail'])} erro(s), cada um com o caminho exato:\n")
for e in r.json()["detail"]:
    print(f"   {'.'.join(map(str, e['loc'])):<28} {e['msg']}")

> 💡 **`itens.0.quantidade`** — o cliente sabe que o problema está no **primeiro** item.
>
> Esse nível de precisão é o que permite a um front-end destacar exatamente o campo com erro no formulário. É de graça.

## 4. 🎯 `response_model` — o contrato de saída

Tão importante quanto validar a entrada é **controlar a saída**.

In [ ]:
class ProdutoBanco(BaseModel):
    """Como o produto vive no banco — inclui dados internos."""
    sku: str
    nome: str
    categoria: str
    preco: float
    custo: float                # 🔴 margem da empresa
    fornecedor: str             # 🔴 informação estratégica
    estoque: int
    observacao_interna: str     # 🔴 nota do comprador


class ProdutoPublico(BaseModel):
    """O que o cliente da API pode ver."""
    sku: str
    nome: str
    categoria: str
    preco: float
    disponivel: bool


BANCO = ProdutoBanco(
    sku="NB-DELL-15", nome="Notebook Dell Inspiron 15", categoria="Notebooks",
    preco=2599.90, custo=2120.00, fornecedor="Distribuidora XYZ Ltda",
    estoque=14, observacao_interna="negociar desconto no próximo lote",
)

app = FastAPI()


@app.get("/produtos/inseguro/{sku}")
def inseguro(sku: str):
    """🔴 Sem response_model: devolve TUDO."""
    return BANCO


@app.get("/produtos/{sku}", response_model=ProdutoPublico)
def seguro(sku: str):
    """✅ O response_model FILTRA a saída."""
    return {**BANCO.model_dump(), "disponivel": BANCO.estoque > 0}


cliente = TestClient(app)
print("🔴 SEM response_model:")
req(cliente, "GET", "/produtos/inseguro/NB-DELL-15")
print("✅ COM response_model:")
req(cliente, "GET", "/produtos/NB-DELL-15")

> 🔴 **Vazamento de dado por resposta é um incidente de segurança real e comum.**
>
> O código de cima não tem bug: ele devolve o objeto que buscou. Mas expõe custo, fornecedor e nota interna para qualquer um que chame a API.
>
> **O `response_model` é a defesa.** Ele não é decoração de documentação — ele **filtra**. Campos que não estão no modelo de saída são removidos, mesmo que você os retorne.
>
> 🧭 **Regra:** toda rota que devolve dado tem `response_model`. Sem exceção.

In [ ]:
# Opções úteis do response_model
class ProdutoOpcional(BaseModel):
    sku: str
    nome: str
    preco: float
    descricao: str | None = None
    tags: list[str] = []


app = FastAPI()


@app.get("/a/{sku}", response_model=ProdutoOpcional)
def com_nulos(sku: str):
    return {"sku": sku, "nome": "Notebook", "preco": 2599.90}


@app.get("/b/{sku}", response_model=ProdutoOpcional,
         response_model_exclude_none=True)
def sem_nulos(sku: str):
    return {"sku": sku, "nome": "Notebook", "preco": 2599.90}


@app.get("/c/{sku}", response_model=ProdutoOpcional,
         response_model_exclude={"tags"})
def sem_tags(sku: str):
    return {"sku": sku, "nome": "Notebook", "preco": 2599.90, "tags": ["a"]}


cliente = TestClient(app)
print("padrão (inclui nulos e vazios):")
req(cliente, "GET", "/a/NB-01")
print("exclude_none:")
req(cliente, "GET", "/b/NB-01")
print("exclude={'tags'}:")
req(cliente, "GET", "/c/NB-01")

### A família de modelos

Um mesmo recurso costuma precisar de **três ou quatro** modelos diferentes.

| Modelo | Para | Diferença |
|--------|------|-----------|
| `ProdutoBase` | Campos comuns | — |
| `ProdutoCriar` | Entrada do `POST` | Sem `id`, sem datas |
| `ProdutoAtualizar` | Entrada do `PATCH` | **Tudo opcional** |
| `ProdutoResposta` | Saída | Com `id` e derivados, **sem** dado interno |

In [ ]:
class ProdutoBase(BaseModel):
    nome: str = Field(min_length=3, max_length=120)
    categoria: str
    preco: float = Field(gt=0)


class ProdutoCriar(ProdutoBase):
    sku: str = Field(min_length=5, pattern=r"^[A-Z]{2}-")
    custo: float = Field(ge=0)
    estoque: int = Field(default=0, ge=0)


class ProdutoAtualizar(BaseModel):
    """PATCH: todo campo é opcional."""
    nome: str | None = Field(default=None, min_length=3)
    categoria: str | None = None
    preco: float | None = Field(default=None, gt=0)
    estoque: int | None = Field(default=None, ge=0)


class ProdutoResposta(ProdutoBase):
    sku: str
    disponivel: bool
    atualizado_em: str


app = FastAPI()
BANCO: dict[str, dict] = {}


@app.post("/produtos", response_model=ProdutoResposta, status_code=201)
def criar(dados: ProdutoCriar):
    if dados.sku in BANCO:
        raise HTTPException(409, "SKU já existe")
    BANCO[dados.sku] = {**dados.model_dump(), "atualizado_em": "2026-08-12T10:00:00"}
    registro = BANCO[dados.sku]
    return {**registro, "disponivel": registro["estoque"] > 0}


@app.patch("/produtos/{sku}", response_model=ProdutoResposta)
def atualizar(sku: str, dados: ProdutoAtualizar):
    if sku not in BANCO:
        raise HTTPException(404, "não encontrado")
    # 🎯 exclude_unset: só os campos que o cliente REALMENTE enviou
    alteracoes = dados.model_dump(exclude_unset=True)
    BANCO[sku].update(alteracoes)
    BANCO[sku]["atualizado_em"] = "2026-08-12T11:30:00"
    registro = BANCO[sku]
    print(f"   [servidor] campos alterados: {list(alteracoes)}")
    return {**registro, "disponivel": registro["estoque"] > 0}


cliente = TestClient(app)
req(cliente, "POST", "/produtos", json={
    "sku": "NB-DELL-15", "nome": "Notebook Dell", "categoria": "Notebooks",
    "preco": 2599.90, "custo": 2120.00, "estoque": 14})

req(cliente, "PATCH", "/produtos/NB-DELL-15", json={"preco": 2399.00})

> 🎯 **`model_dump(exclude_unset=True)` é o que faz o `PATCH` funcionar corretamente.**
>
> Sem ele, os campos não enviados viriam como `None` e sobrescreveriam os valores existentes — o cliente pediu para mudar o preço e apagou o nome.
>
> ⚠️ **A distinção sutil:** `exclude_unset` diferencia "não enviou" de "enviou `null`". Se o cliente mandar `{"descricao": null}` querendo **limpar** o campo, `exclude_unset` preserva essa intenção; `exclude_none` a perderia.

## 5. Tratamento de erros

In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError


class AtlasError(Exception):
    """Exceção base do domínio — a mesma do M01."""
    def __init__(self, mensagem: str, codigo: str = "erro_interno"):
        self.mensagem = mensagem
        self.codigo = codigo
        super().__init__(mensagem)


class RecursoNaoEncontrado(AtlasError):
    def __init__(self, recurso: str, identificador):
        super().__init__(f"{recurso} {identificador} não encontrado", "nao_encontrado")


class EstoqueInsuficiente(AtlasError):
    def __init__(self, sku: str, pedido: int, disponivel: int):
        self.sku, self.pedido, self.disponivel = sku, pedido, disponivel
        super().__init__(
            f"{sku}: pedido {pedido}, disponível {disponivel}", "estoque_insuficiente")


app = FastAPI()


# ── Handlers: traduzem exceção de DOMÍNIO para resposta HTTP ──
@app.exception_handler(RecursoNaoEncontrado)
def tratar_nao_encontrado(requisicao: Request, erro: RecursoNaoEncontrado):
    return JSONResponse(status_code=404,
                        content={"codigo": erro.codigo, "mensagem": erro.mensagem})


@app.exception_handler(EstoqueInsuficiente)
def tratar_estoque(requisicao: Request, erro: EstoqueInsuficiente):
    return JSONResponse(status_code=409, content={
        "codigo": erro.codigo, "mensagem": erro.mensagem,
        "detalhes": {"sku": erro.sku, "solicitado": erro.pedido,
                     "disponivel": erro.disponivel,
                     "faltam": erro.pedido - erro.disponivel},
    })


@app.exception_handler(RequestValidationError)
def tratar_validacao(requisicao: Request, erro: RequestValidationError):
    """Padroniza o 422 no MESMO formato dos outros erros."""
    return JSONResponse(status_code=422, content={
        "codigo": "validacao_falhou",
        "mensagem": "Os dados enviados são inválidos",
        "campos": [{"campo": ".".join(map(str, e["loc"][1:])), "erro": e["msg"]}
                   for e in erro.errors()],
    })


ESTOQUE = {"NB-DELL-15": 3}


@app.get("/produtos/{sku}")
def buscar(sku: str):
    if sku not in ESTOQUE:
        raise RecursoNaoEncontrado("Produto", sku)
    return {"sku": sku, "estoque": ESTOQUE[sku]}


class Reserva(BaseModel):
    quantidade: int = Field(gt=0)


@app.post("/produtos/{sku}/reservas", status_code=201)
def reservar(sku: str, reserva: Reserva):
    if sku not in ESTOQUE:
        raise RecursoNaoEncontrado("Produto", sku)
    if reserva.quantidade > ESTOQUE[sku]:
        raise EstoqueInsuficiente(sku, reserva.quantidade, ESTOQUE[sku])
    ESTOQUE[sku] -= reserva.quantidade
    return {"sku": sku, "reservado": reserva.quantidade, "restante": ESTOQUE[sku]}


cliente = TestClient(app)

req(cliente, "GET", "/produtos/INEXISTENTE")
req(cliente, "POST", "/produtos/NB-DELL-15/reservas", json={"quantidade": 10})
req(cliente, "POST", "/produtos/NB-DELL-15/reservas", json={"quantidade": 0})
req(cliente, "POST", "/produtos/NB-DELL-15/reservas", json={"quantidade": 2})

> 🎯 **Repare no desenho.** As funções de rota levantam exceções **de domínio** — elas não sabem nada de HTTP. Os *handlers* fazem a tradução.
>
> **Por que isso importa:** a mesma função `reservar()` poderia ser chamada por um worker de fila ou por um script CLI. Se ela levantasse `HTTPException`, estaria acoplada ao protocolo web.
>
> É o mesmo princípio das camadas do M04: o domínio não conhece a apresentação.

In [ ]:
# 🔴 Nunca vaze o traceback para o cliente
import traceback

app = FastAPI()


@app.exception_handler(Exception)
def tratar_inesperado(requisicao: Request, erro: Exception):
    """Captura o que ninguém previu."""
    # ✅ o detalhe vai para o LOG (você investiga)
    #    Em produção: logging.error(..., exc_info=erro) — o traceback INTEIRO.
    #    Aqui só medimos o tamanho dele para não poluir a saída do notebook.
    rastro = traceback.format_exception(type(erro), erro, erro.__traceback__)
    print(f"   [log] {type(erro).__name__}: {erro}  "
          f"(+{len(''.join(rastro).splitlines())} linhas de traceback)")

    # ✅ o cliente recebe uma mensagem genérica
    return JSONResponse(status_code=500, content={
        "codigo": "erro_interno",
        "mensagem": "Erro interno. A equipe foi notificada.",
        "id_correlacao": "req-a1b2c3",     # 📌 para o suporte rastrear
    })


@app.get("/quebra")
def quebra():
    return 1 / 0


cliente = TestClient(app, raise_server_exceptions=False)
req(cliente, "GET", "/quebra")

> 🔴 **Traceback numa resposta de API é vazamento de informação.** Ele revela caminhos de arquivo, nomes de biblioteca, versões e às vezes trechos de código e credenciais.
>
> **A regra:** detalhe no log, mensagem genérica na resposta, e um **id de correlação** que liga os dois. Assim o cliente reporta *"deu erro, id req-a1b2c3"* e você encontra a linha exata no log estruturado que construiu no M04.

## 6. Documentando as respostas

O OpenAPI descreve não só o sucesso, mas também os erros possíveis.

In [ ]:
class ErroResposta(BaseModel):
    codigo: str
    mensagem: str


app = FastAPI()


@app.get(
    "/produtos/{sku}",
    response_model=ProdutoPublico,
    summary="Busca um produto",
    responses={
        404: {"model": ErroResposta, "description": "Produto não encontrado"},
        503: {"model": ErroResposta, "description": "Catálogo indisponível"},
    },
    tags=["Produtos"],
)
def buscar(sku: str):
    if sku != "NB-DELL-15":
        raise HTTPException(404, "não encontrado")
    return {"sku": sku, "nome": "Notebook Dell", "categoria": "Notebooks",
            "preco": 2599.90, "disponivel": True}


cliente = TestClient(app)
especificacao = cliente.get("/openapi.json").json()
rota = especificacao["paths"]["/produtos/{sku}"]["get"]

print("Respostas documentadas:")
for codigo, detalhe in sorted(rota["responses"].items()):
    print(f"   {codigo}  {detalhe.get('description', '')}")

## 🔧 Prática guiada — CRUD completo e validado

In [ ]:
from datetime import datetime, timezone

from fastapi import FastAPI, HTTPException, Query

app = FastAPI(title="Atlas API", version="1.1.0")


# ═══════════ Modelos ═══════════
class CategoriaEnum(str, Enum):
    NOTEBOOKS = "Notebooks"
    MONITORES = "Monitores"
    PERIFERICOS = "Periféricos"
    ARMAZENAMENTO = "Armazenamento"


class ProdutoBase(BaseModel):
    nome: str = Field(min_length=3, max_length=120)
    categoria: CategoriaEnum
    preco: float = Field(gt=0, le=1_000_000)

    @field_validator("nome")
    @classmethod
    def limpar(cls, v: str) -> str:
        return " ".join(v.split())


class ProdutoCriar(ProdutoBase):
    sku: str = Field(min_length=5, max_length=20, pattern=r"^[A-Z]{2}-[A-Z0-9-]+$")
    custo: float = Field(ge=0)
    estoque: int = Field(default=0, ge=0)

    @field_validator("sku", mode="before")
    @classmethod
    def maiusculo(cls, v):
        return v.strip().upper() if isinstance(v, str) else v

    @model_validator(mode="after")
    def margem_positiva(self):
        if self.preco < self.custo:
            raise ValueError(f"preço {self.preco} abaixo do custo {self.custo}")
        return self


class ProdutoAtualizar(BaseModel):
    nome: str | None = Field(default=None, min_length=3, max_length=120)
    categoria: CategoriaEnum | None = None
    preco: float | None = Field(default=None, gt=0)
    estoque: int | None = Field(default=None, ge=0)


class ProdutoResposta(ProdutoBase):
    sku: str
    estoque: int
    disponivel: bool
    margem_pct: float
    atualizado_em: datetime


class ListaProdutos(BaseModel):
    total: int
    pagina: int
    por_pagina: int
    itens: list[ProdutoResposta]


class Erro(BaseModel):
    codigo: str
    mensagem: str


# ═══════════ Repositório em memória ═══════════
BANCO: dict[str, dict] = {}


def montar_resposta(registro: dict) -> dict:
    margem = (registro["preco"] - registro["custo"]) / registro["preco"] if registro["preco"] else 0
    return {**registro, "disponivel": registro["estoque"] > 0,
            "margem_pct": round(margem * 100, 1)}


def agora() -> datetime:
    return datetime(2026, 8, 12, 14, 30, tzinfo=timezone.utc)


# ═══════════ Rotas ═══════════
@app.post("/produtos", response_model=ProdutoResposta, status_code=201,
          responses={409: {"model": Erro}}, tags=["Produtos"])
def criar(dados: ProdutoCriar):
    if dados.sku in BANCO:
        raise HTTPException(409, f"SKU {dados.sku} já existe")
    BANCO[dados.sku] = {**dados.model_dump(), "atualizado_em": agora()}
    return montar_resposta(BANCO[dados.sku])


@app.get("/produtos", response_model=ListaProdutos, tags=["Produtos"])
def listar(categoria: CategoriaEnum | None = None,
           limite: int = Query(20, ge=1, le=100),
           pagina: int = Query(1, ge=1)):
    itens = list(BANCO.values())
    if categoria:
        itens = [p for p in itens if p["categoria"] == categoria]
    inicio = (pagina - 1) * limite
    return {"total": len(itens), "pagina": pagina, "por_pagina": limite,
            "itens": [montar_resposta(p) for p in itens[inicio:inicio + limite]]}


@app.get("/produtos/{sku}", response_model=ProdutoResposta,
         responses={404: {"model": Erro}}, tags=["Produtos"])
def buscar(sku: str):
    if sku not in BANCO:
        raise HTTPException(404, f"Produto {sku} não encontrado")
    return montar_resposta(BANCO[sku])


@app.patch("/produtos/{sku}", response_model=ProdutoResposta,
           responses={404: {"model": Erro}}, tags=["Produtos"])
def atualizar(sku: str, dados: ProdutoAtualizar):
    if sku not in BANCO:
        raise HTTPException(404, f"Produto {sku} não encontrado")
    alteracoes = dados.model_dump(exclude_unset=True)
    if not alteracoes:
        raise HTTPException(400, "nenhum campo enviado")
    novo = {**BANCO[sku], **alteracoes}
    if novo["preco"] < novo["custo"]:
        raise HTTPException(422, f"preço {novo['preco']} ficaria abaixo do custo")
    BANCO[sku] = {**novo, "atualizado_em": agora()}
    return montar_resposta(BANCO[sku])


@app.delete("/produtos/{sku}", status_code=204, tags=["Produtos"])
def remover(sku: str):
    if sku not in BANCO:
        raise HTTPException(404, f"Produto {sku} não encontrado")
    del BANCO[sku]


cliente = TestClient(app)
print("✅ CRUD pronto\n")

req(cliente, "POST", "/produtos", json={
    "sku": "nb-dell-15", "nome": "Notebook   Dell  Inspiron 15",
    "categoria": "Notebooks", "preco": 2599.90, "custo": 2120.00, "estoque": 14})

In [ ]:
for produto in [
    {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide", "categoria": "Monitores",
     "preco": 1199.00, "custo": 920.00, "estoque": 31},
    {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170", "categoria": "Periféricos",
     "preco": 89.90, "custo": 52.00, "estoque": 0},
]:
    cliente.post("/produtos", json=produto)

req(cliente, "GET", "/produtos?limite=5")

In [ ]:
print("── PATCH parcial ──")
req(cliente, "PATCH", "/produtos/NB-DELL-15", json={"preco": 2399.00})

print("── PATCH que violaria a margem ──")
req(cliente, "PATCH", "/produtos/NB-DELL-15", json={"preco": 100.0})

print("── PATCH vazio ──")
req(cliente, "PATCH", "/produtos/NB-DELL-15", json={})

In [ ]:
print("── DELETE ──")
req(cliente, "DELETE", "/produtos/PE-LOG-M170", mostrar_corpo=False)
req(cliente, "DELETE", "/produtos/PE-LOG-M170")

print("── conflito de SKU ──")
req(cliente, "POST", "/produtos", json={
    "sku": "MO-LG-24UW", "nome": "Duplicado", "categoria": "Monitores",
    "preco": 100, "custo": 50})

> 💭 **Compare com a API do 06_01.** Ela aceitava qualquer coisa e devolvia dicionários crus.
>
> Esta valida entrada, normaliza texto, aplica regra de negócio, filtra a saída, documenta os erros possíveis e devolve mensagens acionáveis. **E a maior parte disso veio de declarar tipos.**

## 📝 Exercícios

**E1.** Crie `ClienteCriar` com validação de e-mail, CPF (11 dígitos), UF (2 letras maiúsculas) e data de nascimento (maior de 18 anos). Use `field_validator` e `model_validator`.

**E2.** Escreva um `field_validator` que normalize telefone brasileiro para o formato `(19) 99999-9999`, aceitando várias entradas.

**E3.** Modele o corpo de um pedido com endereço aninhado e lista de itens. Valide: ao menos 1 item, no máximo 50, sem SKU repetido, e frete grátis só acima de R$ 500.

**E4.** Demonstre o vazamento de dados: crie um modelo de banco com campos sensíveis, uma rota sem `response_model` e outra com. Compare as respostas.

**E5.** Crie a família completa de modelos para `Cliente`: `Base`, `Criar`, `Atualizar`, `Resposta` e `ResumoLista`. Explique por que são quatro.

**E6.** Implemente um `PATCH` correto com `exclude_unset`. Depois mostre o bug de usar `model_dump()` sem ele.

**E7.** Crie uma hierarquia de exceções de domínio (`AtlasError` e três filhas) e os handlers correspondentes. Prove que as funções de rota não importam nada de `fastapi.HTTPException`.

**E8.** Escreva um handler para `RequestValidationError` que devolva erros no formato `{"campo": "...", "erro": "..."}`, mais amigável para formulários.

**E9.** Escreva um handler global para `Exception` que registre no log e devolva `500` com id de correlação. Prove que o traceback não vaza.

**E10.** Documente as respostas de erro de três rotas com `responses={}`. Verifique no `openapi.json`.

**E11.** Crie `ProdutoResposta` com campos calculados: `disponivel`, `margem_pct` e `faixa_preco`. Use `computed_field` do Pydantic e compare com calcular na rota.

**E12.** Implemente `POST /pedidos` completo: valida o corpo, verifica se todos os SKUs existem, confere estoque de cada item, e devolve `422` com a lista de problemas encontrados.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

## 📋 Cola de referência

```python
from pydantic import BaseModel, Field, field_validator, model_validator

# ── Modelo de entrada ──
class ProdutoCriar(BaseModel):
    sku: str = Field(min_length=5, max_length=20, pattern=r"^[A-Z]{2}-")
    preco: float = Field(gt=0, le=1_000_000)
    estoque: int = Field(default=0, ge=0)
    tags: list[str] = Field(default_factory=list, max_length=10)
    categoria: CategoriaEnum                      # 🔒 domínio fechado
    opcional: str | None = Field(default=None, max_length=500)

    @field_validator("sku", mode="before")        # antes da validação de tipo
    @classmethod
    def normalizar(cls, v): return v.strip().upper()

    @model_validator(mode="after")                # enxerga TODOS os campos
    def regra_cruzada(self):
        if self.preco < self.custo: raise ValueError("...")
        return self

# ── Família de modelos ──
# Base       campos comuns
# Criar      + campos obrigatórios só na criação
# Atualizar  TUDO opcional (PATCH)
# Resposta   + derivados, − dado interno

# ── Rotas ──
@app.post("/produtos", response_model=ProdutoResposta, status_code=201,
          responses={409: {"model": Erro}})
def criar(dados: ProdutoCriar): ...              # modelo → vem do CORPO

@app.patch("/produtos/{sku}", response_model=ProdutoResposta)
def atualizar(sku: str, dados: ProdutoAtualizar):
    alteracoes = dados.model_dump(exclude_unset=True)   # 🎯 só o que enviou
    if not alteracoes: raise HTTPException(400, "nada a alterar")

# ── response_model: FILTRA a saída 🔒 ──
response_model=ProdutoPublico
response_model_exclude_none=True
response_model_exclude={"custo"}

# ── Erros ──
raise HTTPException(status_code=404, detail="...")

@app.exception_handler(MinhaExcecao)
def tratar(requisicao: Request, erro: MinhaExcecao):
    return JSONResponse(status_code=409, content={...})

@app.exception_handler(RequestValidationError)   # padroniza o 422
@app.exception_handler(Exception)                # 🔴 log detalhado, resposta genérica

# ── Serialização ──
modelo.model_dump()                    modelo.model_dump_json()
modelo.model_dump(exclude_unset=True)  # PATCH
modelo.model_dump(exclude={"custo"})
Modelo.model_validate(dicionario)      Modelo.model_validate_json(texto)
```

## ✅ Checklist de saída

- [ ] Sei que modelo Pydantic no parâmetro significa "vem do corpo"
- [ ] Uso `Field()` com restrições em vez de validar na mão
- [ ] Diferencio `field_validator` de `model_validator`
- [ ] Uso validadores para **normalizar**, não só rejeitar
- [ ] Uso `Enum` para domínios fechados
- [ ] Modelo corpos aninhados e leio o `loc` do erro
- [ ] 🔒 **Toda rota que devolve dado tem `response_model`**
- [ ] Entendo que `response_model` **filtra** a saída, não só documenta
- [ ] Crio a família Base/Criar/Atualizar/Resposta
- [ ] **Uso `exclude_unset=True` no `PATCH`**
- [ ] Levanto exceções de domínio, não `HTTPException`, na lógica
- [ ] Escrevo handlers que traduzem domínio → HTTP
- [ ] 🔴 **Nunca vazo traceback na resposta**
- [ ] Uso id de correlação entre resposta e log
- [ ] Documento os erros possíveis com `responses={}`

---

### ➡️ Próxima aula

**`06_03_Arquitetura_e_Banco.ipynb`** — Estrutura de projeto, injeção de dependências e sessão por requisição. Onde a API deixa de ser um arquivo só e passa a conversar com o banco do M05.